# 🎬 Cinescore — Image to Music Generation

Upload an image → extract visual features → generate a matching music track.

**Before you start:** Go to **Runtime → Change runtime type → T4 GPU**

---

| Cell | What it does | Run every session? |
|------|-------------|--------------------|
| 1 | Clone GitHub repo | ✅ Yes |
| 2 | Install dependencies | ✅ Yes |
| 3 | Verify GPU & set device | ✅ Yes |
| 4 | Load all models (~2 min) | ✅ Yes |
| 5 | Upload image & extract features | ✅ Per image |
| 6 | Fuse → MusicSpec + prompt | ✅ Per image |
| 7 | (Optional) Edit the prompt | ⚡ If needed |
| 8 | Generate audio & play | ✅ Per image |
| 9 | Download audio | ⚡ If needed |
| 10 | Re-roll (new seed) | ⚡ If needed |
| 11 | Start web app + get link | ⚡ For web UI |
| 12 | Restart servers (after code changes) | ⚡ If needed |
| 13 | Access from phone via ngrok | ⚡ For mobile |

## Cell 1 — Clone GitHub Repository

In [ ]:
import os
!git clone https://github.com/awsume18-collab/CineScore-AI.git /content/cinescore
os.chdir('/content/cinescore')
print('\n✅ Repository cloned successfully.')


## Cell 2 — Install Dependencies

Handles system libs, audiocraft, and all Python packages in one go.

In [ ]:
# System FFmpeg libraries (needed to build the 'av' package)
!apt-get -qq install -y libavformat-dev libavcodec-dev libavutil-dev \
    libavdevice-dev libavfilter-dev libswscale-dev libswresample-dev pkg-config

# Fix setuptools (torch on Colab needs <82)
!pip install -q "setuptools<82" wheel

# Install av (uses the system FFmpeg libs above)
!pip install -q av

# Install audiocraft without its strict torch==2.1.0 pin
!pip install -q --no-deps git+https://github.com/facebookresearch/audiocraft.git

# All audiocraft runtime dependencies (without version pins)
!pip install -q julius torchdiffeq hydra-core hydra_colorlog torchmetrics \
    omegaconf antlr4-python3-runtime xformers encodec einops flashy \
    lameenc num2words sentencepiece huggingface_hub

# Vision + audio + API + frontend dependencies
!pip install -q transformers open-clip-torch Pillow opencv-python-headless \
    numpy scikit-learn imagehash pyloudnorm soundfile scipy \
    pydantic pydantic-settings fastapi uvicorn streamlit requests jedi

# Verify
from audiocraft.models import MusicGen
print("\n✅ All dependencies installed successfully.")

## Cell 3 — Verify GPU & Set Device

In [ ]:
import os, sys, torch

# Add project to Python path
sys.path.insert(0, "/content/cinescore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU")

os.environ["CINESCORE_DEVICE"] = device
os.environ["CINESCORE_OUTPUT_DIR"] = "/content/cinescore/outputs"
os.makedirs("/content/cinescore/outputs", exist_ok=True)

## Cell 4 — Load All Models (~2 min first time)

In [ ]:
import time

# CLIP
print("Loading CLIP...")
t0 = time.time()
from imgtune.vision.clip_scorer import load_clip_model
clip_model, clip_preprocess, clip_tokenizer, clip_device = load_clip_model(device)
print(f"  ✅ CLIP loaded in {time.time()-t0:.1f}s")

# BLIP-2
print("Loading BLIP-2...")
t0 = time.time()
from imgtune.vision.captioner import load_blip_model
blip_model, blip_processor, blip_device = load_blip_model(device)
print(f"  ✅ BLIP-2 loaded in {time.time()-t0:.1f}s")

# MusicGen
print("Loading MusicGen...")
t0 = time.time()
from imgtune.audio.musicgen import load_musicgen
musicgen_model = load_musicgen(device)
print(f"  ✅ MusicGen loaded in {time.time()-t0:.1f}s")

if device == "cuda":
    print(f"\nVRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("\n🎉 All models ready!")

---

## Cell 5 — Upload an Image & Extract Features

Run this cell each time you want to process a new image.

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display, Audio, HTML
import numpy as np, json, time

# Upload image
print("Upload an image:")
img_upload = files.upload()
img_name = list(img_upload.keys())[0]
image = Image.open(img_name).convert("RGB")
display(image.resize((400, int(400 * image.height / image.width))))

# Extract features
print("\n🔍 Extracting features...")
t0 = time.time()
from imgtune.vision.extract import extract_features
features = extract_features(
    image,
    clip_model=clip_model, clip_preprocess=clip_preprocess,
    clip_tokenizer=clip_tokenizer,
    blip_model=blip_model, blip_processor=blip_processor,
    device=device,
)
print(f"  Done in {time.time()-t0:.1f}s")

# Display results
print(f"\n📸 Caption: {features.caption}")
print("\n📊 Axis scores:")
for axis, scores in features.axes.items():
    print(f"  {axis:15s} → {scores.top:15s} (confidence: {scores.confidence:.3f})")

palette_html = "".join(
    f'<span style="display:inline-block;width:40px;height:40px;'
    f'background:rgb({r},{g},{b});border-radius:6px;margin:3px;"></span>'
    for r, g, b in features.color.dominant_colors
)
display(HTML(f"<b>🎨 Palette:</b> {palette_html}"))
print(f"  Lightness: {features.color.mean_lightness:.2f}  "
      f"Saturation: {features.color.mean_saturation:.2f}  "
      f"Warm ratio: {features.color.warm_ratio:.2f}")

## Cell 6 — Fuse → MusicSpec + Prompt

In [ ]:
from imgtune.mapping.rules import fuse
from imgtune.mapping.prompt_builder import build_prompt

spec = fuse(features)
prompt = build_prompt(spec)

print("🎼 Music Spec:")
print(f"  Genre:       {spec.genre}")
print(f"  Mood:        {', '.join(spec.mood)}")
print(f"  Tempo:       {spec.tempo_bpm} BPM")
print(f"  Key:         {spec.key} {spec.mode}")
print(f"  Instruments: {', '.join(spec.instrumentation)}")
print(f"  Texture:     {spec.texture}")
print(f"  Dynamics:    {spec.dynamics}")
print(f"  Duration:    {spec.duration_s}s")
print(f"  Seed:        {spec.seed}")
print(f"\n📝 Prompt: \"{prompt}\"")

## Cell 7 — (Optional) Edit the Prompt

Uncomment the line below and type your own prompt to override the auto-generated one.

In [ ]:
# prompt = "your custom prompt here"

print(f"Using prompt: \"{prompt}\"")

## Cell 8 — Generate Audio & Play

In [ ]:
print(f"🎵 Generating {spec.duration_s}s of audio (seed={spec.seed})...")
t0 = time.time()

from imgtune.audio.musicgen import generate_audio
wav_tensor, sample_rate = generate_audio(
    prompt, spec.duration_s, spec.seed, model=musicgen_model,
)
print(f"  Generated in {time.time()-t0:.1f}s  |  Sample rate: {sample_rate} Hz")

# Post-process
print("🔧 Post-processing...")
audio_np = wav_tensor.cpu().numpy()
output_path = f"/content/cinescore/outputs/output_{spec.seed}.wav"

from imgtune.audio.postprocess import postprocess
postprocess(audio_np, sample_rate, output_path)
print(f"  ✅ Saved to {output_path}")

# Play
print("\n🎧 Playing audio:")
display(Audio(output_path))

## Cell 9 — Download Audio

In [ ]:
files.download(output_path)
print("✅ Download started!")

## Cell 10 — Re-roll (New Seed, Same Spec)

Generates a different track from the same image & spec.

In [ ]:
import random

new_seed = random.randint(0, 2**31)
print(f"🎲 Re-rolling with seed={new_seed}...")
t0 = time.time()

wav2, sr2 = generate_audio(prompt, spec.duration_s, new_seed, model=musicgen_model)
print(f"  Generated in {time.time()-t0:.1f}s")

reroll_path = f"/content/cinescore/outputs/reroll_{new_seed}.wav"
postprocess(wav2.cpu().numpy(), sr2, reroll_path)

print("\n🎧 Re-rolled audio:")
display(Audio(reroll_path))

---

## Cell 11 — Start Web App + Get Link

Launches FastAPI backend + Streamlit frontend and gives you a browser link.

**Note:** The first image you upload through the web UI will take 2-3 min
(models load inside the API process). Subsequent images are fast (~60s).

In [ ]:
import threading, time, uvicorn, subprocess, sys

sys.path.insert(0, "/content/cinescore")

# Increase API timeout in Streamlit (default 30s is too short for first load)
app_path = "/content/cinescore/web/app.py"
with open(app_path, "r") as f:
    code = f.read()
if "timeout=30" in code:
    code = code.replace("timeout=30", "timeout=300")
    with open(app_path, "w") as f:
        f.write(code)

# Start FastAPI backend (in-process, port 8000)
def run_api():
    uvicorn.run("imgtune.api.main:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()
print("⏳ Starting FastAPI on port 8000...")
time.sleep(5)

import requests
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f"✅ FastAPI running: {r.json()}")
except:
    print("⚠️ FastAPI may still be loading — wait a few seconds, then continue")

# Start Streamlit frontend (port 8501)
subprocess.Popen([
    "streamlit", "run", "web/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "false",
], cwd="/content/cinescore")
time.sleep(4)
print("✅ Streamlit running on port 8501")

# Get browser link via Colab proxy
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"\n🌐 Open this URL in your browser:\n\n   {url}\n")
print("💡 First image upload takes 2-3 min (model loading). After that, ~60s per image.")

## Cell 12 — Restart Servers

**Use this cell when** you edited any project file (e.g. `rules.py`, `prompt_builder.py`)
and need the running servers to pick up the changes. Kills both processes and restarts them.

In [ ]:
# Kill existing server processes
!fuser -k 8000/tcp 2>/dev/null
!fuser -k 8501/tcp 2>/dev/null
!pkill -f uvicorn 2>/dev/null
!pkill -f streamlit 2>/dev/null

import threading, time, uvicorn, subprocess
time.sleep(2)

# Restart FastAPI backend (in-process)
def run_api():
    uvicorn.run("imgtune.api.main:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(5)

# Restart Streamlit frontend
subprocess.Popen([
    "streamlit", "run", "web/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "false",
], cwd="/content/cinescore")

time.sleep(3)
print("✅ Servers restarted! Refresh your browser tab (Ctrl+Shift+R).")

---

## Cell 13 — Access from Phone via ngrok

Creates a public URL so you can open Cinescore on your phone from anywhere.

**First time setup:**
1. Go to [dashboard.ngrok.com](https://dashboard.ngrok.com) and create a free account
2. Copy your **Auth Token** from the dashboard
3. Paste it below replacing `YOUR_AUTHTOKEN`

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

#────────────────────────────────────────────
# PASTE YOUR NGROK AUTH TOKEN BELOW:
ngrok.set_auth_token("YOUR_AUTHTOKEN")
#────────────────────────────────────────────

ngrok.kill()
public_url = ngrok.connect(8501)

print(f"\n🌐 Open this URL on your phone:\n\n   {public_url}\n")
print("(First visit may show a warning page — just click 'Visit Site')")